# aidy.AI — Exploratory Data Analysis (EDA)
### HCLTech Career Shaper Programme | Project 78: Multi-Agent Research Assistant with CrewAI

This notebook conducts reproducible data validation, anomaly detection, statistical profiling, and visual exploratory analysis on the official multi-agent execution log dataset (`dataset.csv`).

**Core Objective:** Understand agent performance metrics (tokens, duration, quality score, error occurrences) across distinct roles (`Researcher`, `Fact_Checker`, `Analyst`, `Writer`, `Editor`, `Publisher`) and research topics.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Ensure project package import path
sys.path.insert(0, os.path.abspath('../src'))

from aidy.data.loader import load_raw_dataset, load_cleaned_dataset
from aidy.data.validation import validate_schema, check_ranges
from aidy.data.metrics import compute_dataset_metrics
from aidy.analytics.eda import run_full_eda
from aidy.analytics.visualizations import apply_matplotlib_theme, PALETTE

apply_matplotlib_theme()
print("Environment initialized successfully.")

## 1. Dataset Loading and Schema Verification
Load the empirical 500-record dataset and verify columns and data types.

In [ ]:
df_raw = pd.read_csv('../data/raw/dataset.csv')
print(f"Raw dataset shape: {df_raw.shape}")
print(f"Columns: {df_raw.columns.tolist()}")
is_valid, issues = validate_schema(df_raw)
print(f"Schema Valid: {is_valid} | Issues: {issues}")
df_raw.head()

## 2. Descriptive Statistics & Summary
Calculate central tendency, dispersion, and role-wise aggregates.

In [ ]:
metrics = compute_dataset_metrics(df_raw)
print(f"Total Runs: {metrics['total_records']}")
print(f"Mean Duration: {metrics['mean_duration_s']}s (Median: {metrics['median_duration_s']}s)")
print(f"Mean Quality: {metrics['mean_quality_score']} (Median: {metrics['median_quality_score']})")
print(f"Mean Tokens: {metrics['mean_tokens_used']} tokens")
print(f"Total Errors: {metrics['total_errors']} across {metrics['runs_with_errors']} runs (Error Rate: {metrics['error_rate']*100:.2f}%)")

role_df = pd.DataFrame(metrics['by_agent_role'])
role_df

## 3. Visualization 1: Average Quality Score by Agent Role
Inspect the baseline performance across autonomous roles.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
role_stats = df_raw.groupby('agent_role')['quality_score'].agg(['mean', 'std', 'count']).sort_values(by='mean', ascending=False)
bars = ax.bar(role_stats.index, role_stats['mean'], yerr=role_stats['std']/np.sqrt(role_stats['count']), capsize=5, color=PALETTE['accent'], edgecolor=PALETTE['border'])
ax.set_ylim(0.5, 1.05)
ax.set_title("Average Quality Score by Agent Role (Mean +/- SEM)", fontsize=12, pad=10, weight='bold')
ax.set_ylabel("Quality Score [0.0 - 1.0]")
ax.set_xlabel("Agent Role")
ax.grid(axis='y', linestyle='--')
plt.tight_layout()
plt.show()

## 4. Visualization 2: Execution Duration vs. Quality Score
Test whether longer runtimes yield higher quality outputs or whether delays correlate with retries.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=150)
clean = df_raw[df_raw['errors'] == 0]
err = df_raw[df_raw['errors'] > 0]

ax.scatter(clean['duration_s'], clean['quality_score'], color=PALETTE['accent_2'], alpha=0.7, s=40, label='Clean Runs (0 Errors)')
ax.scatter(err['duration_s'], err['quality_score'], color=PALETTE['error'], alpha=0.9, s=60, marker='^', label='Runs with Errors (1-2)')

z = np.polyfit(df_raw['duration_s'], df_raw['quality_score'], 1)
p = np.poly1d(z)
xs = np.linspace(df_raw['duration_s'].min(), df_raw['duration_s'].max(), 100)
ax.plot(xs, p(xs), color=PALETTE['accent'], linestyle='--', linewidth=1.8, label=f'Trendline (slope: {z[0]:.4f})')

ax.set_title("Execution Duration vs. Quality Score", fontsize=12, pad=10, weight='bold')
ax.set_xlabel("Duration (s)")
ax.set_ylabel("Quality Score")
ax.grid(True, linestyle='--')
ax.legend(facecolor=PALETTE['surface'], edgecolor=PALETTE['border'])
plt.tight_layout()
plt.show()

## 5. Visualization 3: Token Consumption and Error Distribution
Examine token utilization alongside error occurrence rates.

In [ ]:
fig, ax1 = plt.subplots(figsize=(8.5, 4.5), dpi=150)
role_agg = df_raw.groupby('agent_role').agg(avg_tokens=('tokens_used', 'mean'), error_rate=('errors', lambda x: (x > 0).mean() * 100)).reset_index()
x = np.arange(len(role_agg))
w = 0.38

ax1.bar(x - w/2, role_agg['avg_tokens'], w, label='Avg Tokens', color=PALETTE['accent_2'], edgecolor=PALETTE['border'])
ax1.set_ylabel("Average Tokens Used", color=PALETTE['accent_2'])
ax1.set_xticks(x)
ax1.set_xticklabels(role_agg['agent_role'])
ax1.grid(axis='y', linestyle='--')

ax2 = ax1.twinx()
ax2.bar(x + w/2, role_agg['error_rate'], w, label='Error Rate (%)', color=PALETTE['error'], edgecolor=PALETTE['border'])
ax2.set_ylabel("Error Rate (%)", color=PALETTE['error'])

plt.title("Token Consumption & Error Occurrence by Agent Role", fontsize=12, pad=10, weight='bold')
fig.tight_layout()
plt.show()

## 6. Correlation Matrix & Statistical Takeaways
Check pairwise Pearson correlations between numerical metrics.

In [ ]:
corr = df_raw[['tokens_used', 'duration_s', 'quality_score', 'errors']].corr()
corr